In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Figure 15.9: An illustration of anisotropic curvature: the surface is much steeper in o…

The **condition number** of the Hessian H is:
$$\kappa = \frac{\lambda_{\max}}{\lambda_{\min}}$$

It measures **how anisotropic** the curvature is:
- $\kappa \approx 1$: near-circular contours → GD converges fast
- $\kappa \gg 1$: narrow elongated valley → GD zigzags and converges very slowly

**Stable learning rate requires:** $\alpha \leq 2/\lambda_{\max}$

Number of steps to converge scales as $\mathcal{O}(\kappa)$ for GD.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def make_hessian(l1, l2, theta_deg):
    t = theta_deg * np.pi / 180
    c, s = np.cos(t), np.sin(t)
    R = np.array([[c, -s], [s, c]])
    H = R @ np.diag([l1, l2]) @ R.T
    return H


def gd_path(H, x0, y0, lr, steps=150):
    pos = np.array([x0, y0], dtype=float)
    path = [pos.copy()]
    for _ in range(steps):
        grad = H @ pos
        pos = np.clip(pos - lr * grad, -4, 4)
        path.append(pos.copy())
    return np.array(path)


def draw_condition_number(l1=10.0, l2=1.0, theta_deg=0, lr=0.08, steps=100):
    H = make_hessian(l1, l2, theta_deg)
    kappa = l1 / l2

    f = lambda x, y: 0.5 * (H[0,0]*x**2 + 2*H[0,1]*x*y + H[1,1]*y**2)

    xs = np.linspace(-2.5, 2.5, 200)
    ys = np.linspace(-2.5, 2.5, 200)
    XX, YY = np.meshgrid(xs, ys)
    ZZ = np.clip(f(XX, YY), None, 30)

    path = gd_path(H, -2.0, -1.5, lr, steps)
    losses = np.maximum(1e-10, [f(p[0], p[1]) for p in path])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Contour + path
    axes[0].contourf(XX, YY, ZZ, levels=20, cmap='RdBu_r', alpha=0.5)
    axes[0].contour(XX, YY, ZZ, levels=20, colors='gray', linewidths=0.5, alpha=0.4)
    axes[0].plot(path[:, 0], path[:, 1], '-o', color='#2563eb', ms=3, lw=2, label='GD')
    axes[0].scatter(*path[0], c='#059669', s=80, marker='*', zorder=5, label='Start')
    axes[0].set_xlim(-2.5, 2.5); axes[0].set_ylim(-2.5, 2.5)
    axes[0].set_xlabel('θ₁'); axes[0].set_ylabel('θ₂')
    axes[0].set_title(f'Contours + GD path (α={lr:.3f})', fontsize=10)
    axes[0].legend(fontsize=8)

    # Loss convergence
    axes[1].semilogy(losses, color='#2563eb', lw=2)
    axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss (log scale)')
    axes[1].set_title('Convergence', fontsize=10)
    axes[1].grid(True, which='both', alpha=0.3)

    plt.suptitle(
        f'H = [[{H[0,0]:.2f}, {H[0,1]:.2f}], [{H[1,0]:.2f}, {H[1,1]:.2f}]]  '
        f'λ₁={l1:.1f}, λ₂={l2:.1f}, κ={kappa:.1f}',
        fontsize=10, y=0.0
    )
    plt.tight_layout()
    plt.show()

    stable = lr <= 2 / l1
    print(f'κ = λ_max/λ_min = {l1:.2f}/{l2:.2f} = {kappa:.1f}')
    print(f'Stable α ≤ 2/λ_max = {2/l1:.4f}  |  Your α = {lr:.4f}  → {"✓ stable" if stable else "⚠ may diverge"}')
    print(f'GD convergence ∝ κ = {kappa:.1f}  →  {"⚠ very slow" if kappa>20 else "~ moderate" if kappa>5 else "✓ fast"}')


widgets.interact(
    draw_condition_number,
    l1=widgets.FloatSlider(min=0.5, max=20, step=0.5, value=10, description='λ₁ (steep)', continuous_update=False),
    l2=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1.0, description='λ₂ (flat)', continuous_update=False),
    theta_deg=widgets.IntSlider(min=0, max=90, step=5, value=0, description='Rotation θ°', continuous_update=False),
    lr=widgets.FloatSlider(min=0.001, max=0.3, step=0.005, value=0.08, description='Learning rate α', continuous_update=False),
    steps=widgets.IntSlider(min=20, max=300, step=10, value=100, description='Steps', continuous_update=False),
)